In [98]:
import numpy as np
import matplotlib.pyplot as plt
import refractiveindex as ri


In [99]:
def interface_matrix (n1, n2):

    r = (n1-n2)/(n1+n2)
    t = (2*n1)/(n1+n2)

    return (1/t) * np.array([
        [1, r],
        [r, 1]
    ], dtype=complex)

In [100]:
def propagation_matrix (n, d, wavelength):

    phi = (2*np.pi*n*d)/wavelength

    return np.array([
        [np.exp(-1j*phi), 0],
        [0, np.exp(1j*phi)]
    ], dtype=complex)

In [101]:
def stack_calculation(layers, wavelength_nm):

    M_total = np.eye(2, dtype=complex)
    wavelength_micron = wavelength_nm/1000
    n_prev = 1

    for material, d_layer in layers:
        if hasattr(material, 'get_refractive_index'):
            n_layer = material.get_refractive_index(wavelength_micron)
        else:
            n_layer = material
        M_total = M_total @ interface_matrix(n_prev, n_layer)
        M_total = M_total @ propagation_matrix(n_layer, d_layer, wavelength_nm)
        n_prev = n_layer
        
    n_exit = 1.0
    M_total = M_total @ interface_matrix(n_prev, n_exit)

    r_coeff = M_total[1, 0] / M_total[0, 0]

    R = np.abs(r_coeff)**2
    T = 1 - R
    

    return R, T

In [ ]:
def plot_sensitivity(layers, error_nm=2, iterations=10):
    wavelengths = np.linspace(400, 800, 400)
    plt.figure(figsize=(10, 6))
    for i in range(iterations):
        noisy_layers = []
        for mat, d in layers:
            error = np.random.uniform(-error_nm, error_nm)
            noisy_layers.append((mat, d + error))
        R_list = [stack_calculation(noisy_layers, wl)[0] * 100 for wl in wavelengths]
        plt.plot(wavelengths, R_list, color='red', alpha=0.3)
    plt.title(f"Sensitivity Analysis (Manufacturing Error: ±{error_nm}nm)")
    plt.xlabel("Wavelength (nm)")
    plt.ylabel("Reflectance (%)")
    plt.grid(True, alpha=0.2)
    plt.show()